<a href="https://colab.research.google.com/github/SpencerNGARI/Toxicity_prediction/blob/main/Untitled4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing neccessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, make_scorer)
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb
import lightgbm as lgb



In [ ]:
#loading the dataset
from google.colab import files
uploaded = files.upload()
df = pd.read_csv(next(iter(uploaded)))
df.head()

print(f"Shape : {df.shape}")
print(f"Total features : {df.shape[1]-1}")
print(f"Total samples : {df.shape[0]}")
print(f"Target column : Class")

#Target Class Distribution
print("Target Class Distribution:")
print(df["Class"].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x="Class", data=df)
plt.title('Distribution of Toxicity Class')
plt.show()

#Missing Value Check
print(f"Missing values found:{df.isnull().sum().sum()}")

In [ ]:
#EDA Basic profiling
print("\n Data types:\n", df.dtypes.value_counts())
print("\n Missing values(total):", df.isnull().sum().sum())# initially i found no missing values , where there any I would check for the total.
print("\n Duplicate rows:", df.duplicated().sum())

#drop duplicates. I found no duplicates but I left this in.
df.drop_duplicates(inplace=True)

#unique values
print("\n Unique values:\n", df.nunique)

#summary statistics for numerical columns
print("\n Summary statistics(first 5 features):", df.iloc[:, :5].describe().round(3))


In [ ]:
# Checking for the Class distribution
class_counts = df['Class'].value_counts()
print("\n Class Distribution:")
print(class_counts)
print(f"\n Imbalance ratio: {class_counts.max() / class_counts.min():.2f}:1")

#plots to show distribution
fig, axes = plt.subplots(1,2, figsize=(11,4))

#Bar Chart
axes[0].bar(class_counts.index, class_counts.values, width=0.5)
axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(zip(class_counts.index, class_counts.values)):
    axes[0].text(i, val + 1, str(val), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()


In [ ]:
#Feature Distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
sample_features = numeric_cols[:12]

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(sample_features):
    axes[i].hist(df[col].dropna(), bins=30, color='#4C72B0',
                 edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=8, fontweight='bold')
    axes[i].tick_params(labelsize=7)
plt.suptitle('Feature Distributions (Sample of 12)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight')
plt.show()


In [ ]:
#Skewness Analysis
skewness = df[numeric_cols].skew().sort_values(ascending=False)
high_skew = skewness[abs(skewness) > 2]
print(f"\n Features with |skewness| > 2: {len(high_skew)} out of {len(numeric_cols)}")
print(high_skew.head(15).round(3))

plt.figure(figsize=(10, 4))
skewness.plot(kind='hist', bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
plt.axvline(2, color='red', linestyle='--', label='|skew|=2 threshold')
plt.axvline(-2, color='red', linestyle='--')
plt.title('Distribution of Feature Skewness', fontsize=13, fontweight='bold')
plt.xlabel('Skewness')
plt.ylabel('Number of Features')
plt.legend()
plt.tight_layout()
plt.savefig('skewness_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Zero_Variance Feature Count . This checks how many features have no
# variability so I can dump them and save capacity.
zero_var_cols = [col for col in numeric_cols if df[col].nunique() == 1]
print(f"\n Zero-variance (constant) features: {len(zero_var_cols)} ")
if zero_var_cols:
    print(f"Features with zero variance: {zero_var_cols}")


In [ ]:
#Correlation Analysis
sample_corr_cols = numeric_cols[:50]
corr_matrix = df[sample_corr_cols].corr()

plt.figure(figsize=(14, 12))
#masked to remove mirrored redundant data , to make it easier to read
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, cmap='coolwarm', center=0,
            vmin =-1, vmax=1, linewidths=0.5,cbar_kws={"shrink":0.6}, square=True )
plt.title('Correlation Heatmap (First 50 Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

#Count highly correlated pairs across all features
full_corr = df[numeric_cols].corr().abs()
upper_tri = full_corr.where(np.triu(np.ones(full_corr.shape), k=1).astype(bool))
high_corr_pairs = (upper_tri > 0.9).sum().sum()
print(f"\n Feature pairs with correlation > 0.9: {high_corr_pairs}")



In [ ]:
#Boxplot by the Class
sample_box_cols = numeric_cols[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(sample_box_cols):
    df.boxplot(column=col, by='Class', ax=axes[i],
               boxprops=dict(color='#4C72B0'),
               medianprops=dict(color='#DD8452', linewidth=2))
    axes[i].set_title(col, fontsize=8, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=7)
plt.suptitle('Feature Distributions by Class (Sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_by_class.png', bbox_inches='tight')
plt.show()

In [ ]:
#Preprocessing
le = LabelEncoder()
df['Class_encoded'] = le.fit_transform(df['Class'])

print("\n Label encoding:")
for cls, encoded in zip(le.classes_, le.transform(le.classes_)):
  print(f"{cls}: {encoded}")

X = df.drop(['Class', 'Class_encoded'], axis=1)
y = df['Class_encoded']

#keeping only numeric columns
X = X.select_dtypes(include=[np.number])
print(f"\n Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"Class balnce: {y.value_counts().to_dict()}")



In [ ]:
#removing zero_variance features
var_threshold = VarianceThreshold(threshold=0.01)
X_var = var_threshold.fit_transform(X)
removed_var = X.shape[1] - X_var.shape[1]
kept_cols_var = X.columns[var_threshold.get_support()]
X = pd.DataFrame(X_var, columns=kept_cols_var)

print(f"\n VarianceThreshod (threshold=0.01):")
print(f"Removed {removed_var} features")
print(f" Remaining: {X.shape[1]} features")

In [ ]:
#remove highly correlated featuers to avoid redundancy
def remove_high_correlation(df_feat, threshold=0.95):
  corr_matrix = df_feat.corr().abs()
  upper = corr_matrix.where(
      np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
  )
  to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
  return df_feat.drop(columns=to_drop), to_drop

X_uncorr, dropped_corr = remove_high_correlation(X, threshold=0.95)
print(f"\n Highly correlated features removed:")
print(f" Removed {len(dropped_corr)} features")
print(f" Remaining: {X_uncorr.shape[1]} features")
X = X_uncorr

In [ ]:
#Summary after preprocessing to determine data is in a desired state
print(f" Original features:1203")
print(f" After variance thresholding: {X.shape[1]+len(dropped_corr)}")
print(f" After correlation removal: {X.shape[1]}")
print(f" Final feature count: {X.shape[1]}")

In [ ]:
#Feature Selection
#Finding the univariate filter
K_BEST = min(100, X.shape[1])

select_anova = SelectKBest(score_func=f_classif, k=K_BEST)
select_anova.fit(X,y)

anova_scores = pd.Series(select_anova.scores_, index=X.columns)
anova_top = anova_scores.nlargest(K_BEST)

print(f"\n ANOVA F-score:")
print(anova_top.head(20).round(3))

plt.figure(figsize=(12, 6))
anova_top.head(30).sort_values().plot(kind='barh', color='#4C72B0')
plt.title('Top 30 Features (ANOVA F-score)', fontsize=13, fontweight='bold')
plt.xlabel('ANOVA F-score')
plt.tight_layout()
plt.savefig('anova_top_features.png', bbox_inches='tight')
plt.show()

selected_anova = X.columns[select_anova.get_support()].tolist()


In [ ]:
#Tree-based feature importance to refine
et_clf = ExtraTreesClassifier(n_estimators=200, random_state=42,
                               class_weight='balanced', n_jobs=-1)
et_clf.fit(X, y)

tree_importances = pd.Series(et_clf.feature_importances_, index=X.columns)
tree_top = tree_importances.nlargest(K_BEST)

print(f"\n ExtraTrees Feature Importance — Top 20 Features:")
print(tree_top.head(20).round(5))

plt.figure(figsize=(12, 5))
tree_top.head(30).sort_values().plot(kind='barh', color='#C44E52', edgecolor='white')
plt.title('Top 30 Features by ExtraTrees Importance', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('tree_importance_scores.png', bbox_inches='tight')
plt.show()

selected_tree = tree_top.index.tolist()


In [ ]:
from collections import Counter

TOP_N = 50   # take top N from each method for voting
top_anova_set  = set(anova_scores.nlargest(TOP_N).index)
top_tree_set   = set(tree_importances.nlargest(TOP_N).index)

all_candidates = list(top_anova_set | top_tree_set)
vote_counts = Counter()
for feat in all_candidates:
    votes = sum([feat in top_anova_set, feat in top_tree_set])
    vote_counts[feat] = votes

# Features with 2+ votes
consensus_features = [f for f, v in vote_counts.items() if v >= 2]
print(f"\n Consensus Feature Selection (≥2 of 2 methods, top-{TOP_N} each):")
print(f"   ANOVA top-{TOP_N}   : {len(top_anova_set)} features")
print(f"   Tree top-{TOP_N}    : {len(top_tree_set)} features")
print(f"   Consensus (≥2 votes): {len(consensus_features)} features")

# Sort consensus features by tree importance for interpretability
consensus_features_sorted = (
    tree_importances[consensus_features].sort_values(ascending=False).index.tolist()
)
print(f"\n Final selected features ({len(consensus_features_sorted)}):")
print(consensus_features_sorted[:20], "..." if len(consensus_features_sorted) > 20 else "")

# Build final feature matrix
X_selected = X[consensus_features_sorted]
print(f"\n Final X shape for modeling: {X_selected.shape}")

In [ ]:
#PCA visualizationI because there were too many features.
scaler_pca = RobustScaler()
X_scaled_pca = scaler_pca.fit_transform(X_selected)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled_pca)

plt.figure(figsize=(8, 6))
colors_map = {0: '#4C72B0', 1: '#DD8452'}
labels_map = {0: 'NonToxic', 1: 'Toxic'}
for cls in [0, 1]:
    mask = y == cls
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=colors_map[cls], label=labels_map[cls],
                alpha=0.75, edgecolors='white', s=60)
plt.title(f'PCA (2D) — Explained Variance: {pca.explained_variance_ratio_.sum()*100:.1f}%',
          fontsize=13, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(title='Class')
plt.tight_layout()
plt.savefig('pca_2d.png', bbox_inches='tight')
plt.show()


In [ ]:
#CROss Validation
#i define my cross validation strategy and scoring method
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'roc_auc'  : 'roc_auc',
    'f1_macro' : make_scorer(f1_score, average='macro'),
    'f1_toxic' : make_scorer(f1_score, pos_label=1, average='binary'),
}

X_final = X_selected.values
y_final = y.values

print("\n Cross-Validation Setup:")
print(f"   Strategy  : StratifiedKFold (n_splits=5)")
print(f"   SMOTE     : Applied inside each fold (training data only)")
print(f"   Scaler    : RobustScaler (inside each fold)")
print(f"   Features  : {X_final.shape[1]}")
print(f"   Samples   : {X_final.shape[0]}")


In [ ]:
def evaluate_pipeline(pipeline, name, X, y, cv, scoring):
    """Run cross-validation and return a results dict."""
    results = cross_validate(pipeline, X, y, cv=cv,
                             scoring=scoring, return_train_score=False,
                             n_jobs=-1)
    print(f"\n{'─'*50}")
    print(f"  Model: {name}")
    print(f"{'─'*50}")
    for metric, scores in results.items():
        if metric.startswith('test_'):
            label = metric.replace('test_', '')
            print(f"  {label:<12}: {scores.mean():.4f} ± {scores.std():.4f}")
    return {k.replace('test_', ''): v for k, v in results.items() if k.startswith('test_')}


all_results = {}

In [ ]:
#Logistic Regression
log_reg_pipeline = ImbPipeline([
    ('scaler', RobustScaler()),
    ('smote', SMOTE(random_state=42)),
    ('model', LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced'))
])

log_reg_results = evaluate_pipeline(log_reg_pipeline, 'Logistic Regression', X_final, y_final, CV, scoring)
all_results['Logistic Regression'] = log_reg_results

In [ ]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier
pipe_rf = ImbPipeline([
    ('scaler', RobustScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                      max_features='sqrt', random_state=42, n_jobs=-1))
])
all_results['Random Forest'] = evaluate_pipeline(
    pipe_rf, 'Random Forest', X_final, y_final, CV, scoring)


In [ ]:
#Sopport machine vector
pipe_svm = ImbPipeline([
    ('scaler', RobustScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    SVC(kernel='rbf', C=1.0, gamma='scale',
                   class_weight='balanced', probability=True, random_state=42))
])
all_results['SVM (RBF)'] = evaluate_pipeline(
    pipe_svm, 'SVM (RBF)', X_final, y_final, CV, scoring)



In [ ]:
#XGBoost
scale_pos = (y_final == 0).sum() / (y_final == 1).sum()
pipe_xgb = ImbPipeline([
    ('scaler', RobustScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    xgb.XGBClassifier(n_estimators=300, learning_rate=0.05,
                                  max_depth=4, scale_pos_weight=scale_pos,
                                  use_label_encoder=False,
                                  eval_metric='logloss', random_state=42,
                                  verbosity=0))
])
all_results['XGBoost'] = evaluate_pipeline(
    pipe_xgb, 'XGBoost', X_final, y_final, CV, scoring)


In [ ]:
#Light Gradient Boost Machine
pipe_lgb = ImbPipeline([
    ('scaler', RobustScaler()),
    ('smote',  SMOTE(random_state=42)),
    ('clf',    lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05,
                                   num_leaves=31, class_weight='balanced',
                                   random_state=42, verbose=-1))
])
all_results['LightGBM'] = evaluate_pipeline(
    pipe_lgb, 'LightGBM', X_final, y_final, CV, scoring)

In [ ]:
#Comparing the variuos models
summary_rows = []
for model_name, metrics in all_results.items():
    row = {'Model': model_name}
    for metric, scores in metrics.items():
        row[f'{metric}_mean'] = round(scores.mean(), 4)
        row[f'{metric}_std']  = round(scores.std(),  4)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Model')
summary_df.to_csv('model_comparison.csv')


print("  MODEL COMPARISON SUMMARY (5-Fold Stratified CV + SMOTE)")
display_cols = ['roc_auc_mean', 'roc_auc_std', 'f1_macro_mean', 'f1_toxic_mean']
print(summary_df[display_cols].to_string())



In [ ]:
# Bar chart for the model comparisons
metrics_to_plot = ['roc_auc', 'f1_macro', 'f1_toxic']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, metrics_to_plot):
    means = summary_df[f'{metric}_mean']
    stds  = summary_df[f'{metric}_std']
    bars  = ax.barh(means.index, means.values,
                    xerr=stds.values, color='#4C72B0',
                    edgecolor='white', capsize=4, alpha=0.85)
    ax.set_xlim(0, 1.05)
    ax.set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    ax.set_xlabel('Score')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8)
    for bar, val in zip(bars, means.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)

plt.suptitle('Model Performance Comparison (CV Mean ± Std)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
#Confusion matrix for models prediction correctness
# Finding the best model using ROC AUC Metric
best_model_name = summary_df['roc_auc_mean'].idxmax()
print(f"\n Best model by ROC-AUC: {best_model_name}")

pipeline_map = {
    'Logistic Regression': log_reg_pipeline,
    'Random Forest'            : pipe_rf,
    'SVM (RBF)'                : pipe_svm,
    'XGBoost'                  : pipe_xgb,
    'LightGBM'                 : pipe_lgb,
}
best_pipe = pipeline_map[best_model_name]

# Aggregate confusion matrix
agg_cm = np.zeros((2, 2), dtype=int)
fold_reports = []

for fold, (train_idx, test_idx) in enumerate(CV.split(X_final, y_final)):
    X_tr, X_te = X_final[train_idx], X_final[test_idx]
    y_tr, y_te = y_final[train_idx], y_final[test_idx]
    best_pipe.fit(X_tr, y_tr)
    y_pred = best_pipe.predict(X_te)
    agg_cm += confusion_matrix(y_te, y_pred)
    fold_reports.append(classification_report(y_te, y_pred,
                                               target_names=['NonToxic', 'Toxic'],
                                               output_dict=True))

plt.figure(figsize=(6, 5))
sns.heatmap(agg_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NonToxic', 'Toxic'],
            yticklabels=['NonToxic', 'Toxic'],
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Count'})
plt.title(f'Aggregated Confusion Matrix\n{best_model_name} (5-Fold CV)',
          fontsize=12, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix_best.png', bbox_inches='tight')
plt.show()

In [ ]:
#
avg_precision_toxic = np.mean([r['Toxic']['precision'] for r in fold_reports])
avg_recall_toxic    = np.mean([r['Toxic']['recall']    for r in fold_reports])
avg_f1_toxic        = np.mean([r['Toxic']['f1-score']  for r in fold_reports])

print(f"\n {best_model_name} — Average Metrics for Toxic class across folds:")
print(f"   Precision : {avg_precision_toxic:.4f}")
print(f"   Recall    : {avg_recall_toxic:.4f}")
print(f"   F1-Score  : {avg_f1_toxic:.4f}")


In [ ]:
#SAving the dataset after preprocessing
final_df = pd.DataFrame(X_selected, columns=consensus_features_sorted)
final_df['Class'] = y.values
final_df.to_csv('preprocessed_data.csv', index=False)
print(f"\n Preprocessed dataset saved: preprocessed_data.csv ({final_df.shape})")



In [ ]:
#Save selected feature names
with open('selected_features.txt', 'w') as f:
    for feat in consensus_features_sorted:
        f.write(feat + '\n')
print(f"Selected features saved: selected_features.txt ({len(consensus_features_sorted)} features)")


In [ ]:

print("  PIPELINE COMPLETE — FINAL SUMMARY")

print(f"  Dataset              : {df.shape[0]} samples")
print(f"  Class balance        : NonToxic={int((y==0).sum())} | Toxic={int((y==1).sum())}")
print(f"  Original features    : 1203")
print(f"  After preprocessing  : {X.shape[1]}")
print(f"  After feat. selection: {len(consensus_features_sorted)}")
print(f"  CV strategy          : StratifiedKFold (5 folds) + SMOTE")
print(f"  Best model           : {best_model_name}")
print(f"  Best ROC-AUC         : {summary_df['roc_auc_mean'].max():.4f}")
print(f"  Best F1 (macro)      : {summary_df['f1_macro_mean'].max():.4f}")

print("\n  Output files generated:")
print("  ─ preprocessed_data.csv")
print("  ─ selected_features.txt")
print("  ─ model_comparison.csv")
print("  ─ *.png  (all plots)")
